In [4]:
import pandas as pd

resolved = pd.read_parquet("../data/interim/recipes_resolved_features.parquet")

resolved[
    [
        "RecipeId",
        "RecipeServings",
        "RecipeYield",
        "RecipeServings_clean",
        "servings_parsed_from_yield",
        "ResolvedServings",
        "ResolvedServings_imputed",
        "servings_from_yield",
        "yield_unit_token",
    ]
].sample(20, random_state=18)

,RecipeId,RecipeServings,RecipeYield,RecipeServings_clean,servings_parsed_from_yield,ResolvedServings,ResolvedServings_imputed,servings_from_yield,yield_unit_token
460168,477135,NaN,1 pie,NaN,NaN,NaN,8.0,0,yield__pie
221537,230911,4.0,NaN,4.0,NaN,4.0,4.0,0,
281434,292527,NaN,40 appetizers,NaN,NaN,NaN,8.0,0,
370012,383442,NaN,NaN,NaN,NaN,NaN,4.0,0,
341223,353959,8.0,1 Pan Lasagna,8.0,NaN,8.0,8.0,0,
171258,179179,4.0,NaN,4.0,NaN,4.0,4.0,0,
430642,446605,NaN,NaN,NaN,NaN,NaN,8.0,0,
222476,231883,NaN,1 1/2 cups,NaN,NaN,NaN,6.0,0,yield__cup
389693,403709,12.0,12 bruschetta,12.0,NaN,12.0,12.0,0,
153332,160679,NaN,NaN,NaN,NaN,NaN,6.0,0,


In [5]:
derived_cook = resolved["cooktime_derived_from_total_prep"] == 1

resolved.loc[derived_cook, "CookTime_Minutes_resolved"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

count    82543.000000
mean         0.632531
std         16.161263
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
90%          0.000000
95%          0.000000
99%          0.000000
max       1560.000000
Name: CookTime_Minutes_resolved, dtype: float64

In [6]:
zero_derived_cook = (
    derived_cook
    & (resolved["CookTime_Minutes_resolved"] == 0)
).sum()

total_derived_cook = derived_cook.sum()

print("Derived CookTime count:", total_derived_cook)
print("Derived CookTime = 0:", zero_derived_cook)
print("Percent zero among derived:", zero_derived_cook / total_derived_cook * 100)

Derived CookTime count: 82543
Derived CookTime = 0: 81962
Percent zero among derived: 99.29612444422908


In [7]:
resolved["time_arithmetic_mismatch"].value_counts()

time_arithmetic_mismatch
0    522493
1        24
Name: count, dtype: int64

For 99.30% of recipes where CookTime was missing but could be derived, TotalTime equaled PrepTime. Under the dataset’s time equation, this implies a resolved CookTime of 0 minutes. Therefore, missing CookTime appears to represent recipes with no separately recorded cooking stage, rather than random missingness.

In [12]:
time_cols = [
    "CookTime_Minutes_imputed",
    "PrepTime_Minutes_imputed",
    "TotalTime_Minutes_imputed",
]

resolved[time_cols].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99, 0.995, 0.999]
)

,CookTime_Minutes_imputed,PrepTime_Minutes_imputed,TotalTime_Minutes_imputed
count,5.225170e+05,5.225170e+05,5.225170e+05
mean,1.984317e+02,5.609675e+01,2.545014e+02
std,6.229408e+04,3.095547e+03,6.241279e+04
min,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.000000e+01,1.500000e+01,4.000000e+01
75%,4.500000e+01,2.000000e+01,7.000000e+01
90%,9.000000e+01,3.000000e+01,1.350000e+02
95%,1.800000e+02,6.000000e+01,2.600000e+02
99%,5.400000e+02,3.600000e+02,1.440000e+03
99.5%,1.200000e+03,1.440000e+03,1.540000e+03


In [18]:
resolved.sort_values(
    "TotalTime_Minutes_imputed",
    ascending=False
)[
    [
        "RecipeId",
        "Name",
        "RecipeCategory",
        "CookTime_Minutes_imputed",
        "PrepTime_Minutes_imputed",
        "TotalTime_Minutes_imputed",
        "TotalTime_Minutes",
        "TotalTime"
    ]
].head(20)

,RecipeId,Name,RecipeCategory,CookTime_Minutes_imputed,PrepTime_Minutes_imputed,TotalTime_Minutes_imputed,TotalTime_Minutes,TotalTime
46093,50088,Caroline Cogswell's Celebrated Morning Tonic (...,Beverages,43545600.0,7200.0,43552800.0,43552800.0,PT725880H
276285,287220,Barf Dip,Meat,11358720.0,36000.0,11394720.0,11394720.0,PT189912H
498934,517188,Grilled Iced Apricot Jam,Easy,288000.0,1224000.0,1512000.0,1512000.0,PT25200H
506787,525259,SundaySupper Dates &amp; Leftovers,Low Cholesterol,500.0,1440000.0,1440500.0,1440500.0,PT24008H20M
431970,447963,How to Preserve a Husband,Low Protein,525600.0,525600.0,1051200.0,1051200.0,PT17520H
104344,110019,Miso,Soy/Tofu,240.0,525600.0,525840.0,525840.0,PT8764H
518040,536707,Sweet Garlic Dills,For Large Groups,525600.0,60.0,525660.0,525660.0,PT8761H
106875,112629,Blackberry Wine,Beverages,15.0,388800.0,388815.0,388815.0,PT6480H15M
66937,71511,Cured Pork,Ham,0.0,345600.0,345600.0,345600.0,PT5760H
280499,291571,Homemade Fruit Liquers,Beverages,0.0,288000.0,288000.0,288000.0,PT4800H


In [20]:
nutrition_cols = [
    "SugarContent",
    "CarbohydrateContent",
    "FiberContent",
    "ProteinContent",
    "CholesterolContent",
    "RecipeInstructions",
    "SodiumContent",
    "Calories",
    "SaturatedFatContent",
    "FatContent"
]

resolved[nutrition_cols].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99, 0.995, 0.999]
)

,SugarContent,CarbohydrateContent,FiberContent,ProteinContent,CholesterolContent,SodiumContent,Calories,SaturatedFatContent,FatContent
count,522517.000000,522517.000000,522517.000000,522517.000000,522517.000000,5.225170e+05,522517.000000,522517.000000,522517.000000
mean,21.878254,49.089092,3.843242,17.469510,86.487003,7.672639e+02,484.438580,9.559457,24.614922
std,142.620191,180.822062,8.603163,40.128837,301.987009,4.203621e+03,1397.116649,46.622621,111.485798
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
50%,6.400000,28.200000,2.200000,9.100000,42.600000,3.533000e+02,317.100000,4.700000,13.800000
75%,17.900000,51.100000,4.600000,25.000000,107.900000,7.922000e+02,529.100000,10.800000,27.400000
90%,41.800000,86.800000,8.600000,41.300000,202.300000,1.477200e+03,862.800000,20.300000,49.400000
95%,71.100000,136.800000,12.700000,54.100000,291.500000,2.217100e+03,1305.800000,30.200000,75.600000
99%,299.700000,476.600000,27.100000,95.600000,684.484000,5.569084e+03,3642.672000,84.900000,208.100000
99.5%,430.142000,648.842000,36.842000,123.800000,972.042000,8.765744e+03,5001.530000,123.500000,284.684000


In [23]:
complexity_cols = [
    "NumIngredients",
    "NumQuantities",
    "ResolvedServings",
]

resolved[complexity_cols].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99, 0.995, 0.999] 
)


,NumIngredients,NumQuantities,ResolvedServings
count,522517.000000,522517.000000,339885.000000
mean,7.907211,9.344094,8.604777
std,3.938317,4.058902,114.273123
min,1.000000,0.000000,1.000000
50%,7.000000,9.000000,6.000000
75%,10.000000,12.000000,8.000000
90%,13.000000,15.000000,16.000000
95%,15.000000,17.000000,24.000000
99%,19.000000,21.000000,48.000000
99.5%,21.000000,24.000000,60.000000


Numeric recipe features showed strong right-skewness and extreme upper-tail outliers, especially in nutrition, time, and serving-related variables. Therefore, highly skewed positive variables will be clipped at the 99.5th percentile and transformed using log1p before standardization. Ingredient and quantity counts were only log-transformed because their upper tails were not extreme enough to justify clipping. Binary missingness and derivation indicators are preserved for auditability but will be excluded from the main PCA matrix to avoid making PCA capture data-quality artifacts rather than recipe structure.